# FlutterMind: Fine-Tuning Mistral-7B for Flutter Q&A
**Runtime → Change runtime type → T4 GPU** (before running anything)

---

## Cell 1: Check GPU

In [ ]:
!nvidia-smi

## Cell 2: Clone Repo & Install Dependencies

In [ ]:
!git clone https://github.com/gmayank9999/fluttermind-llm-finetuning.git
%cd fluttermind-llm-finetuning
!pip install -q -r requirements.txt

## Cell 3: Verify Data Files Exist

In [ ]:
import json, os

for f in ['flutter_qa_train.json', 'flutter_qa_val.json', 'flutter_qa_test.json']:
    path = f'data/{f}'
    if os.path.exists(path):
        data = json.load(open(path))
        print(f'{f}: {len(data)} samples ✅')
    else:
        print(f'{f}: NOT FOUND ❌')

## Cell 4: Login to HuggingFace (for Mistral access)
1. Go to https://huggingface.co/settings/tokens
2. Create a token (Read access)
3. Go to https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2 and accept the license
4. Paste your token below

In [ ]:
from huggingface_hub import login
login()  # paste your HF token when prompted

## Cell 5: Train the Model (~ 2-3 hours on T4)
This is the main training cell. Go grab chai ☕

In [ ]:
!python scripts/03_train.py --epochs 3 --batch-size 2 --lr 2e-4

## Cell 6: Verify Training Output

In [ ]:
import os
model_dir = 'outputs/model'
if os.path.exists(model_dir):
    files = os.listdir(model_dir)
    print(f'Model saved with {len(files)} files: ✅')
    for f in files:
        size = os.path.getsize(os.path.join(model_dir, f)) / 1024
        print(f'  {f} ({size:.1f} KB)')
else:
    print('Model directory not found ❌')

## Cell 7: Evaluate - Base vs Fine-Tuned Model (~20 min)

In [ ]:
!python scripts/04_evaluate.py --num_samples 20

## Cell 8: View Evaluation Results

In [ ]:
import json

results_path = 'outputs/model/evaluation_results.json'
if os.path.exists(results_path):
    results = json.load(open(results_path))
    print('=' * 60)
    print('EVALUATION RESULTS')
    print('=' * 60)
    for key, val in results.items():
        if isinstance(val, dict):
            print(f'\n{key}:')
            for k, v in val.items():
                print(f'  {k}: {v}')
        else:
            print(f'{key}: {val}')
else:
    print('Results file not found, check outputs/ directory')
    !find outputs/ -type f

## Cell 9: Inference Demo - Single Questions
Run these one by one and **take screenshots** for your report!

In [ ]:
!python scripts/05_inference.py --mode single --question "What is the difference between StatelessWidget and StatefulWidget in Flutter?"

In [ ]:
!python scripts/05_inference.py --mode single --question "Explain how Provider state management works in Flutter."

In [ ]:
!python scripts/05_inference.py --mode single --question "How do Isolates work in Flutter and when should you use them?"

In [ ]:
!python scripts/05_inference.py --mode single --question "What is the widget lifecycle in Flutter?"

In [ ]:
!python scripts/05_inference.py --mode single --question "How does Navigator 2.0 differ from Navigator 1.0 in Flutter?"

## Cell 10: Download Everything
This zips the outputs folder so you can download it

In [ ]:
!zip -r fluttermind_results.zip outputs/

from google.colab import files
files.download('fluttermind_results.zip')
print('\n✅ Download complete! Extract this zip in your local FlutterMind folder.')

---
## Done! 🎉

**After downloading:**
1. Extract `fluttermind_results.zip` into your local `FlutterMind/` folder
2. Fill the TBD sections in `report/FlutterMind_Experimental_Report.md` with actual numbers
3. Add screenshots from Cell 9 to the report
4. `git add . && git commit -m "results: add training outputs and evaluation results" && git push origin master`